# VisionMirror CatVTON Google Colab GPU Server
This notebook installs CatVTON, starts a native FastAPI model server on GPU, tunnels it via Cloudflare Tunnel, and prints the public API URL.

In [ ]:
# 1. Verify GPU availability
import torch
assert torch.cuda.is_available(), "GPU is not enabled! Please go to Runtime -> Change runtime type and select GPU (T4/A100)."

In [ ]:
# 2. Clone repository & install dependencies
!git clone https://github.com/swarnaverma10/VisionMirror.git
%cd VisionMirror/catvton

# Install standard packages
!pip install -r requirements.txt
!pip install fastapi uvicorn python-multipart

# Robustly install detectron2 and setup densepose import
import torch
import sys
import subprocess
import os
import site

torch_ver = ".".join(torch.__version__.split(".")[:2])
if torch.cuda.is_available() and "+" in torch.__version__:
    cuda_ver = torch.__version__.split("+")[-1]
else:
    cuda_ver = "cpu"

print(f"Detected PyTorch: {torch_ver}, CUDA: {cuda_ver}")

# Step A: Install Detectron2
print("Installing detectron2...")
installed = False

# Try MiroPsota package builder wheels index first
try:
    subprocess.run([
        sys.executable, "-m", "pip", "install", 
        "--extra-index-url", "https://miropsota.github.io/torch_packages_builder",
        "detectron2"
    ], check=True)
    installed = True
    print("Successfully installed detectron2 from community wheel index.")
except Exception as e:
    print(f"Community wheel install failed: {e}")

if not installed:
    # Try official FAIR wheels index
    wheel_url = f"https://dl.fbaipublicfiles.com/detectron2/wheels/{cuda_ver}/torch{torch_ver}/index.html"
    try:
        subprocess.run([
            sys.executable, "-m", "pip", "install", "detectron2",
            "-f", wheel_url
        ], check=True)
        installed = True
        print("Successfully installed detectron2 from official FAIR wheels index.")
    except Exception as e:
        print(f"FAIR wheel install failed: {e}")

if not installed:
    # Fallback to source compilation with proper environment variables
    print("Compiling detectron2 from source (this might take a few minutes)...")
    env = os.environ.copy()
    env["FORCE_CUDA"] = "1"
    env["TORCH_CUDA_ARCH_LIST"] = "8.0;8.6;8.9;9.0"
    try:
        subprocess.run([
            sys.executable, "-m", "pip", "install", 
            "git+https://github.com/facebookresearch/detectron2.git"
        ], env=env, check=True)
        installed = True
        print("Successfully compiled and installed detectron2 from source.")
    except Exception as e:
        print(f"Compilation from source failed: {e}")
        sys.exit(1)

# Step B: Link DensePose to python's site-packages so it's globally importable
print("Setting up DensePose package globally...")
site_pkg = site.getsitepackages()[0]
dest_path = os.path.join(site_pkg, "densepose")

if os.path.exists(dest_path):
    if os.path.islink(dest_path):
        os.unlink(dest_path)
    else:
        import shutil
        shutil.rmtree(dest_path)

os.symlink("/content/VisionMirror/catvton/densepose", dest_path)
print(f"DensePose symlinked successfully to {dest_path}")

# Step C: Verification
print("Verifying installation...")
try:
    import detectron2
    import densepose
    print("SUCCESS: detectron2 and densepose are successfully imported!")
except Exception as e:
    print(f"VERIFICATION FAILED: {e}")
    sys.exit(1)

In [ ]:
# 3. Run FastAPI and Cloudflare Tunnel
import subprocess
import time
import re
import sys

# Download Cloudflare Tunnel client
print("Downloading cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Start FastAPI server in the background
print("Starting FastAPI server...")
server_proc = subprocess.Popen(["uvicorn", "colab_server:app", "--host", "127.0.0.1", "--port", "8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Start Cloudflare Tunnel in the background
print("Starting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(5)  # Wait for startup

# Listen to tunnel output and extract the public URL
public_url = None
try:
    for line in iter(tunnel_proc.stdout.readline, ""):
        sys.stdout.write(line)
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            print("\n" + "="*50)
            print(f"PUBLIC API URL: {public_url}")
            print("="*50 + "\n")
            break
except KeyboardInterrupt:
    print("Stopping...")
    server_proc.terminate()
    tunnel_proc.terminate()

# Keep running to stream logs
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            sys.stdout.write(line)
        time.sleep(0.1)
except KeyboardInterrupt:
    print("Stopping server...")
    server_proc.terminate()
    tunnel_proc.terminate()